# Full Injection Pipeline with Spatially-Varying PSF (GalSim + Rubin deepCoadd)

This notebook runs the complete star cluster injection and recovery pipeline
using **GalSim** for PSF convolution and the **actual spatially-varying CoaddPsf**
from the Rubin deepCoadd exposure.

**Pipeline flow:**
```
Butler deepCoadd
        |
        v
PSF diagnostic  --  measure and validate PSF across image
        |
        v
Injection catalog  --  random positions, magnitudes, sizes
        |
        v
inject_clusters_galsim()  --  actual CoaddPsf at each cluster position
        |
        v
Detection pipeline  --  your detector here
        |
        v
ClusterRetrieval  --  completeness curves
```

**Cells:**
1. Imports
2. Butler data loading
3. PSF diagnostic and validation
4. Injection configuration
5. Generate injection catalog
6. Inject clusters (profile generation + actual PSF convolution)
7. Injection diagnostic plots
8. Detection
9. Retrieval and completeness

---
## 1. Imports

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm

pipeline_path = os.path.expanduser('~/WORK/INJECT/star-cluster-injection-pipeline')
if pipeline_path not in sys.path:
    sys.path.insert(0, pipeline_path)

import galsim
from lsst.daf.butler import Butler
from lsst.geom import Point2D

from src.config import InjectionConfig, ClusterConfig
from src.pipeline import InjectionPipeline
from src.light_profiles import (
    mag_to_flux,
    KingProfile,
    PlummerProfile,
    EFFProfile,
    SersicProfile,
)
from src.retrieval import ClusterRetrieval

plt.style.use('tableau-colorblind10')
%matplotlib inline

print('All imports successful.')

---
## 2. Butler Data Loading

In [ ]:
REPO       = 's3://rubin-dp0-protected/butler.yaml'
COLLECTION = '2.2i/runs/DP0.2'

data_id = {
    'tract': 4431,
    'patch': 17,
    'band':  'i',
}

butler = Butler(REPO, collections=COLLECTION)
coadd  = butler.get('deepCoadd', dataId=data_id)
image  = coadd.image.array.copy()

print(f'Loaded deepCoadd: tract={data_id["tract"]}  patch={data_id["patch"]}  band={data_id["band"]}')
print(f'Image shape : {image.shape}')
print(f'Image dtype : {image.dtype}')
print(f'Image min   : {image.min():.4f}')
print(f'Image max   : {image.max():.4f}')
print(f'Image median: {np.median(image):.4f}')

---
## 3. PSF Diagnostic and Validation

The CoaddPsf must be queried in **focal plane coordinates**, not cutout pixel
coordinates. The bounding box offset is retrieved from the Butler exposure and
added to every position query.

Key point: a deepCoadd cutout does not start at pixel (0, 0) in focal plane
coordinates. Passing raw cutout positions to `computeImage()` will raise
`InvalidPsfError` because those coordinates are outside the range of any
input exposure that went into the coadd.

In [ ]:
# ---- constants ---------------------------------------------------------------
PIXEL_SCALE = 0.2    # arcsec per pixel, standard for Rubin
ZERO_POINT  = 27.0   # AB magnitude zero point

# ---- bounding box offset -----------------------------------------------------
# This is the critical step. The coadd lives at some offset in focal plane
# coordinates. All PSF queries must use focal_x = cutout_x + BBOX_X_MIN.
psf_obj    = coadd.getPsf()
bbox       = coadd.getBBox()
BBOX_X_MIN = bbox.getMinX()
BBOX_Y_MIN = bbox.getMinY()
x_max      = bbox.getMaxX()
y_max      = bbox.getMaxY()
x_mid      = (BBOX_X_MIN + x_max) // 2
y_mid      = (BBOX_Y_MIN + y_max) // 2

print(f'Coadd bounding box : x=[{BBOX_X_MIN}, {x_max}]  y=[{BBOX_Y_MIN}, {y_max}]')
print(f'Coadd centre       : ({x_mid}, {y_mid})')
print(f'Bbox offset stored : BBOX_X_MIN={BBOX_X_MIN}, BBOX_Y_MIN={BBOX_Y_MIN}')
print()

# ---- sample positions in focal plane coords ----------------------------------
N_GRID = 5
xs     = np.linspace(BBOX_X_MIN + 100, x_max - 100, N_GRID).astype(int)
ys     = np.linspace(BBOX_Y_MIN + 100, y_max - 100, N_GRID).astype(int)
XX, YY = np.meshgrid(xs, ys)

grid_fwhm_px = np.full((N_GRID, N_GRID), np.nan)
grid_fwhm_as = np.full((N_GRID, N_GRID), np.nan)

named_positions = {
    'centre'      : (x_mid,            y_mid),
    'top-left'    : (BBOX_X_MIN + 100,  y_max - 100),
    'top-right'   : (x_max - 100,       y_max - 100),
    'bottom-left' : (BBOX_X_MIN + 100,  BBOX_Y_MIN + 100),
    'bottom-right': (x_max - 100,       BBOX_Y_MIN + 100),
}
named_results = {}

# ---- named position table ----------------------------------------------------
print('=== PSF at Named Positions ===')
print(f'{"Position":<15} {"x":>6} {"y":>6} {"FWHM (px)":>10} {"FWHM (arcsec)":>14}  Status')
print('-' * 68)

for name, (x, y) in named_positions.items():
    try:
        shape    = psf_obj.computeShape(Point2D(x, y))
        fwhm_px  = shape.getDeterminantRadius() * 2.355
        fwhm_as  = fwhm_px * PIXEL_SCALE
        status   = 'OK' if 1.5 < fwhm_px < 8.0 else 'SUSPICIOUS'
        named_results[name] = {'x': x, 'y': y, 'fwhm_px': fwhm_px, 'fwhm_as': fwhm_as}
        print(f'{name:<15} {x:>6} {y:>6} {fwhm_px:>10.3f} {fwhm_as:>14.3f}  {status}')
    except Exception as e:
        print(f'{name:<15} {x:>6} {y:>6}  {"ERROR":>10}  {str(e).splitlines()[0]}')
        named_results[name] = None

# ---- full grid measurement ---------------------------------------------------
print()
print(f'Measuring PSF on {N_GRID}x{N_GRID} grid ...')
for i, y in enumerate(ys):
    for j, x in enumerate(xs):
        try:
            shape               = psf_obj.computeShape(Point2D(int(x), int(y)))
            grid_fwhm_px[i, j]  = shape.getDeterminantRadius() * 2.355
            grid_fwhm_as[i, j]  = grid_fwhm_px[i, j] * PIXEL_SCALE
        except Exception:
            pass

valid       = grid_fwhm_px[~np.isnan(grid_fwhm_px)]
PSF_FWHM    = float(np.median(valid))

print()
print(f'Grid PSF statistics ({N_GRID}x{N_GRID} = {N_GRID*N_GRID} positions):')
print(f'  Median : {PSF_FWHM:.4f} px  ({PSF_FWHM * PIXEL_SCALE:.4f} arcsec)')
print(f'  Min    : {valid.min():.4f} px')
print(f'  Max    : {valid.max():.4f} px')
print(f'  Std    : {valid.std():.4f} px  (spatial variation)')
print(f'  N good : {len(valid)} / {N_GRID*N_GRID}')
print()
print(f'PSF_FWHM set to {PSF_FWHM:.3f} px for injection (fallback Gaussian)')

In [ ]:
# ---- PSF diagnostic plots ----------------------------------------------------
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.35)

# -- panel 1: FWHM spatial map -------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])
im1 = ax1.pcolormesh(
    xs - BBOX_X_MIN, ys - BBOX_Y_MIN, grid_fwhm_px,
    cmap='RdYlGn_r', vmin=valid.min() - 0.1, vmax=valid.max() + 0.1
)
plt.colorbar(im1, ax=ax1, label='FWHM (px)')
for i in range(N_GRID):
    for j in range(N_GRID):
        if not np.isnan(grid_fwhm_px[i, j]):
            ax1.text(
                xs[j] - BBOX_X_MIN, ys[i] - BBOX_Y_MIN,
                f'{grid_fwhm_px[i,j]:.2f}',
                ha='center', va='center', fontsize=7,
            )
ax1.set_xlabel('Cutout x (pixels)')
ax1.set_ylabel('Cutout y (pixels)')
ax1.set_title('PSF FWHM Spatial Map')

# -- panel 2: FWHM arcsec spatial map ------------------------------------------
ax2 = fig.add_subplot(gs[0, 1])
im2 = ax2.pcolormesh(
    xs - BBOX_X_MIN, ys - BBOX_Y_MIN, grid_fwhm_as,
    cmap='RdYlGn_r',
)
plt.colorbar(im2, ax=ax2, label='FWHM (arcsec)')
ax2.set_xlabel('Cutout x (pixels)')
ax2.set_ylabel('Cutout y (pixels)')
ax2.set_title('PSF FWHM Spatial Map (arcsec)')

# -- panel 3: FWHM histogram ---------------------------------------------------
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(valid, bins=max(5, len(valid)//2), color='steelblue',
         edgecolor='white', linewidth=0.8)
ax3.axvline(PSF_FWHM, color='tomato', ls='--', lw=1.5,
            label=f'Median = {PSF_FWHM:.3f} px')
ax3.axvline(valid.mean(), color='darkorange', ls=':', lw=1.5,
            label=f'Mean   = {valid.mean():.3f} px')
ax3.set_xlabel('FWHM (pixels)')
ax3.set_ylabel('Count')
ax3.set_title('PSF FWHM Distribution')
ax3.legend(fontsize=9)

# -- panel 4: PSF image at centre ----------------------------------------------
ax4   = fig.add_subplot(gs[1, 0])
psf_c = psf_obj.computeImage(Point2D(x_mid, y_mid))
psf_a = psf_c.array
im4   = ax4.imshow(psf_a, origin='lower', cmap='magma')
plt.colorbar(im4, ax=ax4, label='Normalised flux')
ax4.set_title(
    f'PSF Image at Centre\n'
    f'FWHM = {named_results["centre"]["fwhm_px"]:.3f} px'
)
ax4.set_xlabel('x (pixels)')
ax4.set_ylabel('y (pixels)')

# -- panel 5: PSF radial profile -----------------------------------------------
ax5        = fig.add_subplot(gs[1, 1])
cy_p, cx_p = np.array(psf_a.shape) // 2
y_idx, x_idx = np.indices(psf_a.shape)
r          = np.sqrt((x_idx - cx_p)**2 + (y_idx - cy_p)**2).ravel()
flux_r     = psf_a.ravel()
r_sort     = np.argsort(r)

ax5.scatter(r[r_sort], flux_r[r_sort], s=4, alpha=0.4,
            color='steelblue', label='All pixels')

# Binned median radial profile
r_bins   = np.arange(0, r.max(), 0.5)
r_centres = 0.5 * (r_bins[:-1] + r_bins[1:])
r_median  = [
    np.median(flux_r[(r >= r_bins[k]) & (r < r_bins[k+1])])
    for k in range(len(r_bins)-1)
]
ax5.plot(r_centres, r_median, color='tomato', lw=2, label='Binned median')
ax5.axvline(
    named_results['centre']['fwhm_px'] / 2, ls='--',
    color='darkorange', lw=1.5,
    label=f'Half-FWHM = {named_results["centre"]["fwhm_px"]/2:.2f} px'
)
ax5.set_xlabel('Radius (pixels)')
ax5.set_ylabel('Normalised flux')
ax5.set_title('PSF Radial Profile at Centre')
ax5.legend(fontsize=8)

# -- panel 6: FWHM variation along image diagonal ------------------------------
ax6 = fig.add_subplot(gs[1, 2])
diag_fwhm = [grid_fwhm_px[i, i] for i in range(N_GRID)
             if not np.isnan(grid_fwhm_px[i, i])]
diag_pos  = [
    np.sqrt((xs[i] - BBOX_X_MIN)**2 + (ys[i] - BBOX_Y_MIN)**2)
    for i in range(N_GRID)
    if not np.isnan(grid_fwhm_px[i, i])
]
ax6.plot(diag_pos, diag_fwhm, 'o-', color='steelblue',
         lw=2, ms=6, label='FWHM along diagonal')
ax6.axhline(PSF_FWHM, ls='--', color='tomato', lw=1.5,
            label=f'Median = {PSF_FWHM:.3f} px')
ax6.fill_between(
    [min(diag_pos), max(diag_pos)],
    PSF_FWHM - valid.std(), PSF_FWHM + valid.std(),
    alpha=0.2, color='tomato', label='1 sigma range'
)
ax6.set_xlabel('Distance from corner (pixels)')
ax6.set_ylabel('FWHM (pixels)')
ax6.set_title('PSF Spatial Variation Along Diagonal')
ax6.legend(fontsize=8)

fig.suptitle(
    f'CoaddPsf Diagnostic -- tract {data_id["tract"]}  '
    f'patch {data_id["patch"]}  {data_id["band"]}-band\n'
    f'Median FWHM = {PSF_FWHM:.3f} px ({PSF_FWHM*PIXEL_SCALE:.3f} arcsec)  '
    f'Spatial variation = {valid.std():.4f} px',
    fontsize=13,
)
plt.savefig('psf_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()
print('PSF diagnostic plot saved to psf_diagnostic.png')

---
## 4. Injection Configuration

In [ ]:
cluster_cfg = ClusterConfig(
    profile_type      = 'king',
    method            = 'smooth',
    mag_min           = 20.0,
    mag_max           = 26.0,
    r_half_min        = 2.0,
    r_half_max        = 10.0,
    concentration_min = 5.0,
    concentration_max = 30.0,
)

config = InjectionConfig(
    run_name           = 'demo_galsim_actual_psf',
    n_clusters         = 100,
    seed               = 42,
    edge_buffer        = 50,
    add_noise          = True,
    use_actual_psf     = True,
    save_injected_image= False,
    cluster_config     = cluster_cfg,
    tract              = data_id['tract'],
    patch              = data_id['patch'],
    band               = data_id['band'],
    output_dir         = 'outputs/demo_galsim_actual_psf',
)

os.makedirs(config.output_dir, exist_ok=True)
print(config)
print(f'PSF_FWHM (measured, used as fallback) : {PSF_FWHM:.3f} px')
print(f'PIXEL_SCALE                           : {PIXEL_SCALE} arcsec/px')
print(f'ZERO_POINT                            : {ZERO_POINT}')

---
## 5. Generate Injection Catalog

In [ ]:
pipeline = InjectionPipeline(config)
pipeline.load_data(image=image)
catalog  = pipeline.generate_catalog()

print(f'Generated catalog with {len(catalog)} entries.')
print()
print(f'{"id":>4}  {"x":>7}  {"y":>7}  {"mag":>6}  {"r_half":>7}  {"conc":>6}')
print('-' * 48)
for entry in catalog[:10]:
    print(
        f'{entry["id"]:>4}  '
        f'{entry["x"]:>7.1f}  '
        f'{entry["y"]:>7.1f}  '
        f'{entry["magnitude"]:>6.2f}  '
        f'{entry["r_half"]:>7.2f}  '
        f'{entry.get("concentration", 0):>6.1f}'
    )
print('  ... (first 10 shown)')

---
## 6. Inject Clusters

For each cluster the pipeline:
1. Generates a 2D light-profile stamp using our `light_profiles.py` classes
2. Converts the cutout position to focal plane coordinates
3. Fetches the actual CoaddPsf kernel at that position
4. Wraps both as GalSim objects and convolves
5. Falls back to a Gaussian PSF if the fetch fails at any position

In [ ]:
def make_profile_image(entry, pixel_scale=PIXEL_SCALE):
    """
    Generate a 2D light-profile stamp for one catalog entry using
    the classes in src/light_profiles.py.

    Parameters
    ----------
    entry : dict
        Must contain 'r_half', 'magnitude', 'profile_type'.
        King and EFF also use 'concentration'.
    pixel_scale : float
        arcsec per pixel.

    Returns
    -------
    profile_arr : 2D ndarray (float64)
        Normalised to unit total flux (the flux scaling is done later).
    stamp_size  : int
        Side length of the stamp (odd number).
    """
    r_half       = entry['r_half']             # pixels
    magnitude    = entry['magnitude']
    profile_type = entry.get('profile_type', 'king').lower()
    conc         = entry.get('concentration', 10.0)

    # Choose stamp size: ~20x the half-light radius, at least 31, at most 257
    stamp_size = int(np.clip(20 * r_half, 31, 257))
    if stamp_size % 2 == 0:
        stamp_size += 1

    shape  = (stamp_size, stamp_size)
    # Use age=1.0 as a dummy; it does not affect the profile shape
    age    = 1.0

    if profile_type == 'king':
        prof = KingProfile(
            r_half=r_half, concentration=conc, age=age,
            magnitude=magnitude, zero_point=ZERO_POINT,
        )
    elif profile_type == 'plummer':
        prof = PlummerProfile(
            r_half=r_half, age=age,
            magnitude=magnitude, zero_point=ZERO_POINT,
        )
    elif profile_type == 'eff':
        gamma = conc if conc > 2.01 else 2.5
        prof = EFFProfile(
            r_half=r_half, gamma=gamma, age=age,
            magnitude=magnitude, zero_point=ZERO_POINT,
        )
    elif profile_type == 'sersic':
        sersic_n = conc if conc > 0.3 else 1.0
        prof = SersicProfile(
            r_half=r_half, sersic_n=sersic_n, age=age,
            magnitude=magnitude, zero_point=ZERO_POINT,
        )
    else:
        raise ValueError(f'Unknown profile type: {profile_type}')

    profile_arr = prof.generate_2d(shape).astype(np.float64)

    # Normalise to unit flux (caller will rescale to mag-based flux)
    arr_sum = profile_arr.sum()
    if arr_sum > 0:
        profile_arr /= arr_sum

    return profile_arr, stamp_size


# Quick sanity check
_test = make_profile_image(catalog[0])
print(f'make_profile_image test: stamp shape={_test[0].shape}, '
      f'sum={_test[0].sum():.6f}, stamp_size={_test[1]}')

In [ ]:
def get_psf_at_position(psf_obj, cutout_x, cutout_y,
                         bbox_x_min, bbox_y_min, pixel_scale):
    """
    Return the CoaddPsf as a GalSim InterpolatedImage at a given
    cutout pixel position.

    Parameters
    ----------
    psf_obj    : lsst PSF object from coadd.getPsf()
    cutout_x   : float  -- x position in cutout pixel coords (0-indexed)
    cutout_y   : float  -- y position in cutout pixel coords (0-indexed)
    bbox_x_min : int    -- coadd bounding box x offset
    bbox_y_min : int    -- coadd bounding box y offset
    pixel_scale: float  -- arcsec per pixel

    Returns
    -------
    psf_gs     : galsim.InterpolatedImage
    fwhm_px    : float  -- PSF FWHM at this position in pixels
    """
    focal_x   = cutout_x + bbox_x_min
    focal_y   = cutout_y + bbox_y_min
    point     = Point2D(focal_x, focal_y)

    psf_image = psf_obj.computeImage(point)
    psf_array = psf_image.array.astype(np.float64)
    psf_sum   = psf_array.sum()
    if psf_sum > 0:
        psf_array /= psf_sum

    gs_img    = galsim.Image(psf_array, scale=pixel_scale)
    psf_gs    = galsim.InterpolatedImage(gs_img, normalization='flux')

    shape     = psf_obj.computeShape(point)
    fwhm_px   = shape.getDeterminantRadius() * 2.355

    return psf_gs, fwhm_px


def inject_clusters_galsim(image, catalog, psf_fwhm,
                            pixel_scale  = PIXEL_SCALE,
                            add_noise    = True,
                            rng_seed     = 42,
                            psf_obj      = None,
                            bbox_x_min   = 0,
                            bbox_y_min   = 0):
    """
    Inject all catalog entries into image using the actual CoaddPsf.

    If psf_obj is provided the spatially-varying PSF is fetched at each
    cluster position. If psf_obj is None or the fetch fails at a given
    position, falls back to a Gaussian with the given psf_fwhm.
    """
    ny, nx           = image.shape
    injected         = image.copy().astype(np.float64)
    rng_np           = np.random.default_rng(rng_seed)
    injection_info   = []

    psf_fwhm_arcsec  = psf_fwhm * pixel_scale
    gaussian_psf     = galsim.Gaussian(fwhm=psf_fwhm_arcsec)
    using_actual_psf = psf_obj is not None

    print(f'  PSF mode   : {"actual spatially-varying CoaddPsf" if using_actual_psf else "Gaussian fallback"}')
    print(f'  N clusters : {len(catalog)}')
    print()

    n_ok      = 0
    n_failed  = 0
    n_psf_fallback = 0

    for i, entry in enumerate(catalog):
        try:
            cx = int(round(entry['x']))
            cy = int(round(entry['y']))

            # Step 1: cluster profile image (normalised to unit flux)
            profile_arr, stamp_size = make_profile_image(entry, pixel_scale)

            # Step 2: wrap as GalSim InterpolatedImage
            gs_img     = galsim.Image(profile_arr, scale=pixel_scale)
            cluster_gs = galsim.InterpolatedImage(gs_img, normalization='flux')

            # Step 3: get PSF at this cluster position
            if using_actual_psf:
                try:
                    psf_here, fwhm_here = get_psf_at_position(
                        psf_obj, cx, cy, bbox_x_min, bbox_y_min, pixel_scale
                    )
                except Exception as psf_err:
                    psf_here    = gaussian_psf
                    fwhm_here   = psf_fwhm
                    n_psf_fallback += 1
                    if n_psf_fallback <= 5:
                        print(f'  PSF fallback at ({cx},{cy}): {str(psf_err).splitlines()[0]}')
            else:
                psf_here  = gaussian_psf
                fwhm_here = psf_fwhm

            # Step 4: convolve
            convolved = galsim.Convolve([cluster_gs, psf_here])

            # Step 5: draw stamp
            out_img = galsim.Image(stamp_size, stamp_size, scale=pixel_scale)
            convolved.drawImage(image=out_img, method='no_pixel')
            stamp = out_img.array.copy().astype(np.float64)

            # Step 6: scale to correct total flux from magnitude
            total_flux = mag_to_flux(entry['magnitude'], zero_point=ZERO_POINT)
            stamp_sum  = stamp.sum()
            if stamp_sum > 0:
                stamp *= total_flux / stamp_sum

            # Step 7: Poisson-like noise on the source
            if add_noise:
                noise_sigma = np.sqrt(np.clip(stamp, 0, None))
                stamp += rng_np.normal(
                    0.0, np.where(noise_sigma > 0, noise_sigma, 1e-10)
                )

            # Step 8: add into image with boundary clipping
            sh, sw = stamp.shape
            y0 = cy - sh // 2;  y1 = y0 + sh
            x0 = cx - sw // 2;  x1 = x0 + sw

            iy0 = max(y0, 0);  iy1 = min(y1, ny)
            ix0 = max(x0, 0);  ix1 = min(x1, nx)

            if iy0 >= iy1 or ix0 >= ix1:
                continue

            sy0 = iy0 - y0;  sy1 = sy0 + (iy1 - iy0)
            sx0 = ix0 - x0;  sx1 = sx0 + (ix1 - ix0)

            injected[iy0:iy1, ix0:ix1] += stamp[sy0:sy1, sx0:sx1]

            info                 = dict(entry)
            info['stamp']        = stamp
            info['stamp_flux']   = float(stamp.sum())
            info['total_flux']   = total_flux
            info['psf_fwhm_px']  = fwhm_here
            injection_info.append(info)
            n_ok += 1

        except Exception as exc:
            n_failed += 1
            if n_failed <= 10:
                print(f'  Cluster {i} (id={entry.get("id","?")}) failed: {exc}')

    if n_psf_fallback > 5:
        print(f'  ... ({n_psf_fallback} total PSF fallbacks, only first 5 shown)')
    if n_failed > 10:
        print(f'  ... ({n_failed} total failures, only first 10 shown)')

    print(f'\nInjection complete.')
    print(f'  Successful         : {n_ok}')
    print(f'  Failed             : {n_failed}')
    print(f'  PSF fallback used  : {n_psf_fallback}')

    return injected.astype(image.dtype), injection_info


print('Injection functions defined.')

In [ ]:
print(f'Injecting {len(catalog)} clusters ...')
print(f'  Profile : {config.cluster_config.profile_type}')
print(f'  PSF     : actual CoaddPsf (fallback Gaussian FWHM={PSF_FWHM:.2f} px)')
print()

injected_image, injection_info = inject_clusters_galsim(
    image,
    catalog,
    psf_fwhm   = PSF_FWHM,
    pixel_scale= PIXEL_SCALE,
    add_noise  = config.add_noise,
    rng_seed   = config.seed,
    psf_obj    = psf_obj,
    bbox_x_min = BBOX_X_MIN,
    bbox_y_min = BBOX_Y_MIN,
)

print(f'\nSuccessfully injected {len(injection_info)} / {len(catalog)} clusters.')

---
## 7. Injection Diagnostic Plots

In [ ]:
# ---- before / after / difference ---------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmin, vmax = np.percentile(image, [1, 99])
kw = dict(cmap='gray', origin='lower', vmin=vmin, vmax=vmax)

axes[0].imshow(image, **kw)
axes[0].set_title('Original')

axes[1].imshow(injected_image, **kw)
axes[1].set_title('Injected')

diff = injected_image.astype(np.float64) - image.astype(np.float64)
dv   = np.percentile(np.abs(diff), 99)
axes[2].imshow(diff, cmap='RdBu_r', origin='lower', vmin=-dv, vmax=dv)
axes[2].set_title('Difference')

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'before_after.png'), dpi=150)
plt.show()

In [ ]:
# ---- stamp grid --------------------------------------------------------------
n_show = min(20, len(injection_info))
n_cols = 5
n_rows = int(np.ceil(n_show / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3 * n_rows))
axes_flat = axes.flat if n_rows > 1 else [axes] if n_show == 1 else axes

for idx, ax in enumerate(axes_flat):
    if idx < n_show:
        info  = injection_info[idx]
        stamp = info['stamp']
        sv1   = np.percentile(stamp, 1)
        sv2   = np.percentile(stamp, 99)
        ax.imshow(stamp, origin='lower', cmap='magma', vmin=sv1, vmax=sv2)
        ax.set_title(
            f'm={info["magnitude"]:.1f}  r={info["r_half"]:.1f}px\n'
            f'psf={info["psf_fwhm_px"]:.2f}px',
            fontsize=6.5,
        )
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(
    f'Injected Cluster Stamps (actual CoaddPsf convolved)  --  '
    f'{config.cluster_config.profile_type.upper()} profile',
    fontsize=13,
)
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'stamp_grid.png'), dpi=150)
plt.show()

In [ ]:
# ---- PSF FWHM distribution at injection positions ----------------------------
inj_fwhms = [info['psf_fwhm_px'] for info in injection_info]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(inj_fwhms, bins=20, color='steelblue', edgecolor='white', lw=0.8)
ax.axvline(np.median(inj_fwhms), ls='--', color='tomato', lw=1.5,
           label=f'Median = {np.median(inj_fwhms):.3f} px')
ax.set_xlabel('PSF FWHM at injection position (pixels)')
ax.set_ylabel('Count')
ax.set_title('Distribution of PSF FWHM Across Injection Positions')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'injection_psf_fwhm.png'), dpi=150)
plt.show()

In [ ]:
# ---- injected parameter distributions ----------------------------------------
mags  = [info['magnitude'] for info in injection_info]
rhs   = [info['r_half']    for info in injection_info]
concs = [info.get('concentration', np.nan) for info in injection_info]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(mags, bins=20, color='steelblue', edgecolor='k')
axes[0].set_xlabel('Magnitude'); axes[0].set_ylabel('N')

axes[1].hist(rhs, bins=20, color='darkorange', edgecolor='k')
axes[1].set_xlabel('r_half (pix)')

axes[2].hist(concs, bins=20, color='seagreen', edgecolor='k')
axes[2].set_xlabel('Concentration')

plt.suptitle('Injected Cluster Parameters', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'injected_params.png'), dpi=150)
plt.show()

---
## 8. Detection

Run your detection pipeline here and produce a list of dicts.
Each dict must have at minimum keys `'x'` and `'y'`.

Below is a placeholder using SEP (uncomment to use), or replace with your
own detection code.

In [ ]:
# Replace this cell with your detection pipeline.
# The only requirement is that `detections` is a list of dicts
# with at minimum keys 'x' and 'y'.
#
# Example using SEP:
#
#   import sep
#   from astropy.stats import sigma_clipped_stats
#   _, median_bg, std_bg = sigma_clipped_stats(injected_image)
#   data    = (injected_image - median_bg).astype(np.float64)
#   objects = sep.extract(data, 3.0, err=std_bg)
#   detections = [
#       {'x': float(o['x']), 'y': float(o['y']),
#        'flux': float(o['flux']), 'r_half': float(o['a']),
#        'snr': float(o['peak'] / std_bg), 'flag': int(o['flag'])}
#       for o in objects
#   ]
#
# For now we use a placeholder:
detections = []
print('Replace this cell with your detection pipeline.')
print(f'detections must be a list of dicts with at minimum keys x and y.')
print(f'Currently: {len(detections)} detections (placeholder).')

---
## 9. Retrieval and Completeness

These cells will produce meaningful results once the detection cell above
is populated.

In [ ]:
MATCH_RADIUS = 5.0  # pixels

if len(detections) == 0:
    print('No detections yet -- populate the detection cell above first.')
    print('Skipping retrieval and completeness.')
else:
    retrieval = ClusterRetrieval(injection_info, detections)
    retrieval.match_detections(match_radius=MATCH_RADIUS)
    stats     = retrieval.get_summary_statistics()

    print('=== Retrieval Summary ===')
    print(f'  Injected         : {stats["n_injected"]}')
    print(f'  Recovered        : {stats.get("n_recovered", stats.get("n_detected", 0))}')
    print(f'  Completeness     : {stats["overall_completeness"]:.1%}')
    print(f'  50% mag limit    : {stats.get("mag_50_limit", float("nan")):.2f}')

In [ ]:
# ---- completeness vs magnitude -----------------------------------------------
if len(detections) > 0:
    mag_bins = np.linspace(config.cluster_config.mag_min,
                           config.cluster_config.mag_max, 15)

    try:
        bc_mag, comp_mag, comp_mag_err = retrieval.compute_completeness(
            parameter='magnitude', bins=mag_bins
        )
    except (TypeError, AttributeError):
        # Fallback for older retrieval API
        comp_mag = retrieval.completeness_vs_magnitude(mag_bins)
        bc_mag   = 0.5 * (mag_bins[:-1] + mag_bins[1:])
        comp_mag_err = np.zeros_like(comp_mag)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.step(bc_mag, comp_mag, where='mid', lw=2, color='royalblue')
    if comp_mag_err is not None and np.any(comp_mag_err > 0):
        ax.fill_between(bc_mag, comp_mag - comp_mag_err, comp_mag + comp_mag_err,
                        step='mid', alpha=0.3, color='royalblue')
    ax.axhline(0.5, ls='--', color='gray', lw=1.2)
    ax.axhline(0.9, ls=':',  color='gray', lw=1.2)
    ax.set_xlabel('Injected Magnitude', fontsize=13)
    ax.set_ylabel('Completeness',       fontsize=13)
    ax.set_ylim(-0.05, 1.15)
    ax.set_title('Completeness vs Magnitude', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'completeness_vs_mag.png'), dpi=150)
    plt.show()
else:
    print('Skipping completeness plot (no detections).')

In [ ]:
# ---- completeness vs half-light radius ---------------------------------------
if len(detections) > 0:
    rh_bins = np.linspace(config.cluster_config.r_half_min,
                          config.cluster_config.r_half_max, 10)

    try:
        bc_rh, comp_rh, comp_rh_err = retrieval.compute_completeness(
            parameter='r_half', bins=rh_bins
        )
    except (TypeError, AttributeError):
        comp_rh = retrieval.completeness_vs_size(rh_bins)
        bc_rh   = 0.5 * (rh_bins[:-1] + rh_bins[1:])
        comp_rh_err = np.zeros_like(comp_rh)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.step(bc_rh, comp_rh, where='mid', lw=2, color='darkorange')
    if comp_rh_err is not None and np.any(comp_rh_err > 0):
        ax.fill_between(bc_rh, comp_rh - comp_rh_err, comp_rh + comp_rh_err,
                        step='mid', alpha=0.3, color='darkorange')
    ax.axhline(0.5, ls='--', color='gray', lw=1.2)
    ax.set_xlabel('Half-light Radius (pixels)', fontsize=13)
    ax.set_ylabel('Completeness',               fontsize=13)
    ax.set_ylim(-0.05, 1.15)
    ax.set_title('Completeness vs Half-light Radius', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'completeness_vs_rh.png'), dpi=150)
    plt.show()
else:
    print('Skipping completeness plot (no detections).')

In [ ]:
# ---- 2D completeness map (magnitude x half-light radius) --------------------
if len(detections) > 0:
    mag_bins_2d = np.linspace(config.cluster_config.mag_min,
                              config.cluster_config.mag_max, 10)
    rh_bins_2d  = np.linspace(config.cluster_config.r_half_min,
                               config.cluster_config.r_half_max, 8)

    try:
        result_2d = retrieval.compute_completeness_2d(
            param1='magnitude', param2='r_half',
            bins1=mag_bins_2d,  bins2=rh_bins_2d,
        )
        comp_2d  = result_2d['completeness']
        bc_mag2  = result_2d['bin_centers1']
        bc_rh2   = result_2d['bin_centers2']
        n_inj_2d = result_2d['n_injected']
    except (TypeError, AttributeError):
        comp_2d  = retrieval.completeness_2d(mag_bins_2d, rh_bins_2d)
        bc_mag2  = 0.5 * (mag_bins_2d[:-1] + mag_bins_2d[1:])
        bc_rh2   = 0.5 * (rh_bins_2d[:-1] + rh_bins_2d[1:])
        n_inj_2d = None

    fig, axes = plt.subplots(1, 2 if n_inj_2d is not None else 1,
                             figsize=(15 if n_inj_2d is not None else 8, 5))
    if n_inj_2d is None:
        axes = [axes]

    im = axes[0].pcolormesh(mag_bins_2d, rh_bins_2d, comp_2d.T,
                             cmap='RdYlGn', vmin=0, vmax=1)
    plt.colorbar(im, ax=axes[0], label='Completeness')
    axes[0].set_xlabel('Injected Magnitude',        fontsize=12)
    axes[0].set_ylabel('Half-light Radius (pixels)', fontsize=12)
    axes[0].set_title('2D Completeness Map',         fontsize=13)

    if n_inj_2d is not None:
        im2 = axes[1].pcolormesh(mag_bins_2d, rh_bins_2d, n_inj_2d.T, cmap='Blues')
        plt.colorbar(im2, ax=axes[1], label='N injected')
        axes[1].set_xlabel('Injected Magnitude',        fontsize=12)
        axes[1].set_ylabel('Half-light Radius (pixels)', fontsize=12)
        axes[1].set_title('N Injected per Cell',         fontsize=13)

    plt.suptitle(
        f'Completeness Analysis -- {config.cluster_config.profile_type.upper()} profile',
        fontsize=14,
    )
    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'completeness_2d.png'), dpi=150)
    plt.show()
else:
    print('Skipping 2D completeness map (no detections).')

In [ ]:
print('\n=== Pipeline complete ===')
print(f'Outputs saved to: {config.output_dir}/')
print(f'  Injected clusters : {len(injection_info)}')
print(f'  Detections        : {len(detections)}')